# TransformerDecoder

从结构上看, `TransformerDecoder` 和前面的 `TransformerEncoder` 很像: 都是先用 `Embedding` 将离散 token 映射到连续向量空间, 然后堆叠多个由注意力层, 前馈网络, 残差连接, LayerNorm 和 Dropout 组成的 block, 最后用线性层输出 logits.

Encoder 使用的是双向自注意力. 对一个输入序列而言, 注意力分数仍然来自:

$$\frac{QK^T}{\sqrt{d_k}}$$

只要不是 padding, 第 `i` 个 token 理论上可以和任意第 `j` 个 token 建立关系. 也就是说, Encoder 的注意力矩阵是完整可见的 token-to-token 关系矩阵. 它适合做句子分类, 序列理解, 表征抽取这类任务, 因为模型在处理每个位置时可以同时利用左侧和右侧上下文.

Decoder 的核心变化是引入因果约束. 在语言模型里, 第 `i` 个 token 只能根据它之前的 token 预测下一个 token, 不能提前看到未来答案. 所以 Decoder 在 softmax 之前会把注意力分数矩阵的右上三角部分 mask 掉:

$$score_{i,j}=-\infty, \quad j>i$$

这样 softmax 后, 未来位置的权重会变成 0. 第 `i` 行不再代表“从整句所有 token 汇聚信息”, 而是代表“只能从当前位置及其之前的 token 汇聚信息”. 这就是 causal self-attention, 也是自回归生成能够成立的根本原因.

因此, Encoder 和 Decoder 的关键区别可以理解为:

1. Encoder 学习的是完整上下文中的双向关系, 适合理解任务.

2. Decoder 学习的是带时间方向约束的单向关系, 适合生成任务.

3. Encoder 通常用 `attention_mask` 屏蔽 padding; Decoder 除了可能需要屏蔽无效位置, 还必须使用 causal mask 屏蔽未来 token.

本 notebook 中的 Decoder 更准确地说是 GPT 风格的 decoder-only 模型. 它没有原始 Transformer Decoder 中用于读取 Encoder 输出的 cross-attention 层, 只有带因果 mask 的 self-attention. 所以它是一个直接用于 next-token prediction 的自回归语言模型.

为了保持教学直观实现, 本章采用普通 padding batch. 一个 batch 保持 `(batch, seq_len)` 形状, 短序列在右侧补 `<pad>`, 训练时用 `attention_mask` 屏蔽 padding token. 因此当前 attention 中实际有两层约束: 一层是不能看未来的 causal mask, 一层是不能看 padding 的 `attention_mask`.


In [1]:
import torch
from torch import nn

torch.manual_seed(0)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(d_model, d_model)

    def _rotate_half(self, x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)
    
    def _rope(self, q, k, base=10000):
        seq_len = q.size(-2)
        freqs = torch.arange(0, self.d_k, 2, device=q.device) / self.d_k
        freqs = base ** -freqs
        angles = torch.arange(seq_len, device=q.device)[:, None] * freqs[None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[None, None]
        cos = angles.cos().repeat_interleave(2, dim=-1)[None, None]
        return q * cos + self._rotate_half(q) * sin, \
              k * cos + self._rotate_half(k) * sin

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape

        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        Q, K = self._rope(Q, K)
        score = Q @ K.transpose(-2, -1) / self.d_k ** 0.5

        causal_mask = torch.triu( # 加入 causal mask, 屏蔽右上三角
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1
        )
        mask_value = torch.finfo(score.dtype).min
        score = score.masked_fill(causal_mask[None, None, :, :], mask_value)

        if attention_mask is not None:
            key_mask = attention_mask[:, None, None, :] == 0
            score = score.masked_fill(key_mask, mask_value)

        attn_w = torch.softmax(score, dim=-1)
        output = self.dropout(attn_w) @ V
        output = output.transpose(1, 2).reshape(batch_size, -1, self.d_model)
        output = self.out(output)
        return output


class TransformerDecoderBlock(nn.Module):
    def __init__(self, ffn_dim, d_model, num_heads, dropout):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attention_mask=None):
        x_norm = self.norm1(x)
        x = x + self.dropout(self.attn(x_norm, attention_mask=attention_mask))
        x_norm = self.norm2(x)
        x = x + self.dropout(self.ffn(x_norm))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, output_dim, ffn_dim, d_model, num_heads, num_layers, dropout=0.1, padding_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(ffn_dim, d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, output_dim)

    def forward(self, x, attention_mask=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask)
        return self.out(self.norm(x))

In [2]:
from datasets import load_dataset

batch_size = 64
subset_size = 30_000
max_length = 256

dataset = load_dataset("larryvrh/Chinese-Poems")
subset = dataset["train"].select(range(min(subset_size, len(dataset["train"]))))

def format_poem(poem):
    # 按照 标题 作者 朝代 内容 拼接成语言模型要学习的一整段文本.
    return {"text": f"{poem["title"]} {poem["author"]} {poem["dynasty"]}\n{poem["content"]}"}

text_dataset = subset.map(format_poem, desc="Formatting poems")
splits = text_dataset.train_test_split(test_size=0.05, seed=0)
train_dataset = splits["train"]
val_dataset = splits["test"]

train_texts = train_dataset["text"]
text_lengths = [len(text) for text in train_texts]
unique_chars = sorted(set("".join(train_texts)))

print(f"训练样本数: {len(train_texts)}, 验证样本数: {len(val_dataset)}")
print(f"最大文本长度: {max(text_lengths)}")
print(f"最小文本长度: {min(text_lengths)}")
print(f"字符表大小(不含特殊token): {len(unique_chars)}")
print(f"超过 max_length={max_length} 的样本比例: {sum(length + 1 > max_length for length in text_lengths) / len(text_lengths):.2%}")

训练样本数: 28500, 验证样本数: 1500
最大文本长度: 16513
最小文本长度: 13
字符表大小(不含特殊token): 9191
超过 max_length=256 的样本比例: 2.14%


中文古诗天然以单字为基本语义单元, 且换行符 `\n` 和标点(`，`, `。`)直接决定诗句结构与节奏. 所以这里我们手写一个字符分词器.

我们还需要一些特殊 token 表示特殊情况

- `<pad>` (`id=0`): 批处理对齐填充
- `<eos>` (`id=1`): 每首诗结尾的终止符, 推理时遇到它就停止生成
- `<unk>` (`id=2`): 验证或推理阶段遇到未登录字时的兜底 token

In [3]:
class CharTokenizer:
    def __init__(self, chars):
        self.id_to_token = ["<pad>", "<eos>", "<unk>"] + list(chars)
        self.id_to_text = ["", "", ""] + list(chars)
        self.token_to_id = {token: idx for idx, token in enumerate(self.id_to_token)}
        self.pad_token_id = 0
        self.eos_token_id = 1
        self.unk_token_id = 2
        self.vocab_size = len(self.id_to_token)

    def __call__(self, texts, max_length):
        return {"input_ids": [self.encode(text, max_length) for text in texts]}

    def encode(self, text, max_length):
        ids = [self.token_to_id.get(char, self.unk_token_id) for char in text[:max_length - 1]]
        return ids + [self.eos_token_id]

    def decode(self, ids):
        return "".join(self.id_to_text[int(idx)] for idx in ids)

tokenizer = CharTokenizer(unique_chars)
print(f"vocab_size: {tokenizer.vocab_size}")

vocab_size: 9194


不同诗句长度差异很大, 如果把每首诗都补齐到 `max_length=256`, 大量位置会是无意义的 `<pad>`.

因此 `collate_fn` 只把每个 batch 补齐到当前 batch 内最长样本的长度也就是动态 padding.

In [4]:
from torch.utils.data import DataLoader

def tokenize_fn(examples):
    return tokenizer(examples["text"], max_length)

train_tokenized = train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)

val_tokenized = val_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing val"
)

def collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros_like(input_ids)

    for i, item in enumerate(batch):
        ids = torch.tensor(item["input_ids"], dtype=torch.long)
        input_ids[i, :len(ids)] = ids
        attention_mask[i, :len(ids)] = 1

    return {
        "input_ids": input_ids,
        "labels": input_ids.clone(),
        "attention_mask": attention_mask,
    }

train_dataloader = DataLoader(
    train_tokenized,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

val_dataloader = DataLoader(
    val_tokenized,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

first_batch = next(iter(train_dataloader))
print({key: tuple(value.shape) for key, value in first_batch.items()})
print(f"train batches: {len(train_dataloader)}, val batches: {len(val_dataloader)}")

{'input_ids': (64, 235), 'labels': (64, 235), 'attention_mask': (64, 235)}
train batches: 446, val batches: 24


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.vocab_size
model = TransformerDecoder(
    vocab_size=vocab_size,
    output_dim=vocab_size,
    ffn_dim=256,
    d_model=128,
    num_heads=8,
    num_layers=6,
    dropout=0.1,
    padding_idx=tokenizer.pad_token_id,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

print(f"device: {device}")
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

device: cuda
parameters: 3.16M


In [6]:
epochs = 2

for i in range(epochs):
    model.train()
    train_loss = 0.0
    train_tokens = 0
    for idx, data in enumerate(train_dataloader):
        input_ids = data["input_ids"].to(device)
        labels = data["labels"].to(device)
        attention_mask = data["attention_mask"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask=attention_mask)
        target = labels[:, 1:].reshape(-1)
        loss = criterion(
            logits[:, :-1, :].reshape(-1, logits.size(-1)),
            target,
        )

        loss.backward()
        optimizer.step()

        valid = (target != tokenizer.pad_token_id).sum().item()
        train_loss += loss.item() * valid
        train_tokens += valid

        if (idx+1) % 50 == 0:
            print(f"Epoch {i+1}, Step {idx+1}, Loss: {loss.item():.4f}, PPL: {torch.exp(loss).item():.4f}")

    model.eval()
    val_loss = 0.0
    val_tokens = 0
    with torch.no_grad():
        for data in val_dataloader:
            input_ids = data["input_ids"].to(device)
            labels = data["labels"].to(device)
            attention_mask = data["attention_mask"].to(device)

            logits = model(input_ids, attention_mask=attention_mask)
            target = labels[:, 1:].reshape(-1)
            loss = criterion(
                logits[:, :-1, :].reshape(-1, logits.size(-1)),
                target,
            )

            valid = (target != tokenizer.pad_token_id).sum().item()
            val_loss += loss.item() * valid
            val_tokens += valid
        
    print(f"Epoch {i+1}, Train Loss: {train_loss / train_tokens:.4f}, Val Loss: {val_loss / val_tokens:.4f}")

Epoch 1, Step 50, Loss: 6.3553, PPL: 575.5270
Epoch 1, Step 100, Loss: 5.9770, PPL: 394.2469
Epoch 1, Step 150, Loss: 5.8120, PPL: 334.2799
Epoch 1, Step 200, Loss: 5.8569, PPL: 349.6532
Epoch 1, Step 250, Loss: 5.5644, PPL: 260.9750
Epoch 1, Step 300, Loss: 5.5492, PPL: 257.0298
Epoch 1, Step 350, Loss: 5.5608, PPL: 260.0420
Epoch 1, Step 400, Loss: 5.5576, PPL: 259.2053
Epoch 1, Train Loss: 5.8575, Val Loss: 5.3546
Epoch 2, Step 50, Loss: 5.2717, PPL: 194.7466
Epoch 2, Step 100, Loss: 5.2657, PPL: 193.5857
Epoch 2, Step 150, Loss: 5.2174, PPL: 184.4519
Epoch 2, Step 200, Loss: 5.3556, PPL: 211.7964
Epoch 2, Step 250, Loss: 5.1598, PPL: 174.1224
Epoch 2, Step 300, Loss: 5.0436, PPL: 155.0345
Epoch 2, Step 350, Loss: 4.9647, PPL: 143.2703
Epoch 2, Step 400, Loss: 5.0407, PPL: 154.5810
Epoch 2, Train Loss: 5.1589, Val Loss: 4.9906


推理时模型按自回归方式每次只预测 1 个新字, 取序列最后一位的 `logits[:, -1, :]`, 经 `temperature` 缩放和 `top_k` 截断后按概率采样, 拼回输入序列循环往复, 直到生成 `<eos>`
或达到 `max_new_tokens`.

In [7]:

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens, temperature, top_k, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    prompt_ids = tokenizer.encode(prompt, max_length=len(prompt) + 1)[:-1]
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    
    for _ in range(max_new_tokens):
        logits = model(input_ids)
        next_logits = logits[:, -1, :] / temperature
        
        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            next_logits = torch.full_like(next_logits, float('-inf'))
            next_logits.scatter_(1, indices, values)
        
        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        input_ids = torch.cat([input_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    generated_text = tokenizer.decode(input_ids[0])
    return generated_text

prompts = [
    "浣溪沙 李白 唐代\n",
    "静夜思 李白 唐代\n",
    "春望 杜甫 唐代\n",
]
for prompt in prompts:
    print(generate(model, tokenizer, prompt, max_new_tokens=max_length, temperature=0.8, top_k=20))
    print("=" * 40)


浣溪沙 李白 唐代
一年年有主人生，一见西风不见秋。
夜风风雨无人去，一日无声不可怜。
静夜思 李白 唐代
东苑春光好，风流水落归。
何处一时候，还自一番归。
一片山前路，何时有酒心。
春望 杜甫 唐代
山下黄昏雪，东风独见微。
夜飞天色下，云水有花风。
水落云深夜，烟霞自不开。
春去不为酒，秋声有客心。
